# Debug Viewer Notebook (v8.0 - Get URL First)
Strategy: **Get URL First**.
To avoid `eval_js` hangs or JS scope issues, we retrieve the Proxy URL **before** starting any server processes.
Then we display it as a simple clickable link.

In [ ]:
import os
import sys
import subprocess
import time
from google.colab.output import eval_js
from IPython.display import display, Markdown

# 1. Get Proxy URL FIRST (Port 8000)
# We do this before running any subprocess to ensure the kernel is responsive.
print("Resolving Proxy URL for port 8000...")
try:
    proxy_url = eval_js("google.colab.kernel.proxyPort(8000)")
    print(f"Got URL: {proxy_url}")
except Exception as e:
    print(f"Error getting URL: {e}")
    proxy_url = None

# 2. Setup Server
viewer_dir = '/content/viewer'
if not os.path.exists(viewer_dir):
    subprocess.run(["git", "clone", "https://github.com/antimatter15/splat", viewer_dir])

with open(f"{viewer_dir}/simple_server.py", 'w') as f:
    f.write('''
from http.server import HTTPServer, SimpleHTTPRequestHandler
import sys
class CORSRequestHandler(SimpleHTTPRequestHandler):
    def end_headers(self):
        self.send_header('Cross-Origin-Opener-Policy', 'same-origin')
        self.send_header('Cross-Origin-Embedder-Policy', 'require-corp')
        self.send_header('Access-Control-Allow-Origin', '*')
        super().end_headers()
if __name__ == '__main__':
    HTTPServer(('', 8000), CORSRequestHandler).serve_forever()
''')

# 3. Start Server
print("Starting Server...")
subprocess.run(["fuser", "-k", "8000/tcp"])
proc = subprocess.Popen([sys.executable, "simple_server.py", "8000"], cwd=viewer_dir, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

if proc.poll() is None:
    print("Server is running.")
    if proxy_url:
        full_url = f"{proxy_url}index.html"
        print(f"\nSUCCESS! Click the link below to open the viewer:")
        display(Markdown(f"## [OPEN 3DGS VIEWER]({full_url})"))
    else:
        print("Server running, but could not resolve URL.")
else:
    print("Server failed to start.")